In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,MinMaxScaler,PowerTransformer
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn import set_config
from lightgbm import LGBMRegressor
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

import optuna
import dagshub
import mlflow

c:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# load data

df = pd.read_csv(r'C:\Users\Jay Kanakia\Desktop\CampusX\Projects\swiggy_delivery_time_prediction\notebooks\data_eda.csv')
df.sample(2)

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
13017,INDORES06DEL02,37.0,4.5,22.728163,75.884212,22.748163,75.904212,2022-03-26,cloudy,high,...,INDO,26,3,saturday,1,15.0,11.0,morning,3.025321,short
16045,RANCHIRES13DEL02,39.0,4.7,23.374989,85.335486,23.424989,85.385486,2022-03-13,cloudy,low,...,RANCHI,13,3,sunday,1,10.0,23.0,night,7.546265,medium


In [5]:
set_config(transform_output='pandas')

In [6]:
df.isnull().sum()

rider_id                   0
age                     1854
ratings                 1908
restaurant_latitude     3630
restaurant_longitude    3630
delivery_latitude       3630
delivery_longitude      3630
order_date                 0
weather                  525
traffic                  510
vehicle_condition          0
type_of_order              0
type_of_vehicle            0
multiple_deliveries      993
festival                 228
city_type               1198
time_taken                 0
city_name                  0
order_day                  0
order_month                0
order_day_of_week          0
is_weekend                 0
pickup_time_minutes     1640
order_time_hour         1640
order_time_of_day       2070
distance                3630
distance_type           3630
dtype: int64

In [7]:
# dropping columns
df.drop(columns=['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"],
                    inplace=True)

df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
44132,37.0,4.8,windy,low,2,meal,scooter,0.0,no,urban,18,0,5.0,night,20.206901,very_long
24592,35.0,4.8,sandstorms,low,2,drinks,scooter,0.0,no,metropolitian,27,1,10.0,morning,1.554499,short


## Model Building

In [8]:
temp_df = df.copy().dropna()
temp_df.sample(2)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
35750,29.0,4.6,windy,medium,2,drinks,motorcycle,1.0,no,urban,18,0,15.0,evening,10.711186,long
19953,26.0,4.9,sunny,low,2,buffet,scooter,0.0,no,metropolitian,21,0,5.0,night,4.544232,short


In [9]:
temp_df.isnull().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [10]:
X = temp_df.drop(columns=['time_taken'])
y = temp_df['time_taken']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape,y_test.shape

((30156, 15), (7539,))

In [11]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [22]:
# Transforming target column

pt = PowerTransformer(method='yeo-johnson')

y_train_pt = pt.fit_transform(y_train.values.reshape(-1,1))
y_test_pt = pt.transform(y_test.values.reshape(-1,1))

y_train_pt


,x0
0,2.028672
1,0.554539
2,-2.024267
3,-0.173699
4,0.554539
...,...
30151,0.457580
30152,-0.173699
30153,-1.350937
30154,0.047111


In [13]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [14]:
# build a preprocessor

preprocessor = ColumnTransformer(
    transformers=
    [
        ("scale", MinMaxScaler(), num_cols),
        ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",sparse_output=False), nominal_cat_cols),
        ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],handle_unknown="use_encoded_value",unknown_value=-1), ordinal_cat_cols)
    ],
        remainder="passthrough",n_jobs=-1,verbose_feature_names_out=False
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [15]:
# build the pipeline

processing_pipeline = Pipeline(steps=[
                                ("preprocess",preprocessor)
                            ])

processing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transfor

In [16]:
X_train_trans = processing_pipeline.fit_transform(X_train)
X_test_trans = processing_pipeline.transform(X_test)

## Optuna HP tuning

In [17]:
dagshub.init(repo_owner='jay-kanakia', repo_name='swiggy_delivery_time_prediction', mlflow=True)
mlflow.set_tracking_uri('https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow')
mlflow.set_experiment(experiment_name='Exp 4 : RF Optuna HF tuning')

Accessing as jay-kanakia

Initialized MLflow to track repo "jay-kanakia/swiggy_delivery_time_prediction"

Repository jay-kanakia/swiggy_delivery_time_prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/f78ff6a2bd624df5b3400b1f635740f5', creation_time=1775393931096, experiment_id='5', last_update_time=1775393931096, lifecycle_stage='active', name='Exp 4 : RF Optuna HF tuning', tags={}, workspace='default'>

In [23]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            'n_estimators' : trial.suggest_int('n_estimators',10,500),
            'max_depth' : trial.suggest_int('max_depth',1,30),
            'max_features' : trial.suggest_categorical('max_features',[None,'sqrt','log2']),
            'min_samples_split' : trial.suggest_int('min_samples_split',2,10),
            'min_samples_leaf' : trial.suggest_int('min_samples_leaf',1,10),
            'max_samples' : trial.suggest_float('max_samples',0.5,1),
            'random_state' : 42,
            'n_jobs' : -1
        }

        # log model parameters
        mlflow.log_params(params)

        # build the model
        rfg = RandomForestRegressor(**params)
        rfg.fit(X_train_trans,y_train)

        # perform cross validation
        model = TransformedTargetRegressor(regressor=rfg,transformer=pt)

        # perform cross validation
        score  = cross_val_score(model,X_train_trans,y_train,cv=5,scoring='neg_mean_absolute_error',n_jobs=-1)

        # mean score
        mean_score  = -score.mean()

        # get the prediction
        y_pred_train = rfg.predict(X_train_trans)
        y_pred_test = rfg.predict(X_test_trans)

        # logging metrics
        mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train))
        mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test))
        mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train))
        mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test))
        mlflow.log_metric("cross_val",mean_score)

        return mean_score

In [24]:
# create optuna study
study = optuna.create_study(direction="minimize",study_name='Optuna Light GBM HP')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=50,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # build the model on best parameters
    best_rf = RandomForestRegressor(**study.best_params)
    best_rf.fit(X_train_trans,y_train)

    # get the predictions
    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)

    # get the actual predictions values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


    #train the model
    model = TransformedTargetRegressor(regressor=best_rf,transformer=pt)
    
    # perform cross validation
    scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")

[I 2026-04-05 21:26:57,543] A new study created in memory with name: Optuna Light GBM HP
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run worried-wasp-314 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5/runs/3eec3ea33bcb42ce939ca80a81c5e7b8
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5
🏃 View run ambitious-crane-519 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5/runs/57d16d3f3b004d56979f6c36b38999f1
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5


Best trial: 4. Best value: 3.09158:   2%|▏         | 1/50 [01:02<50:59, 62.44s/it]

[I 2026-04-05 21:28:01,324] Trial 4 finished with value: 3.0915787506101453 and parameters: {'n_estimators': 233, 'max_depth': 25, 'max_features': None, 'min_samples_split': 9, 'min_samples_leaf': 6, 'max_samples': 0.5192638383577522}. Best is trial 4 with value: 3.0915787506101453.
🏃 View run resilient-shrimp-723 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5/runs/d38b95d6908c40e68eabd38057e9f658
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5


Best trial: 4. Best value: 3.09158:   4%|▍         | 2/50 [01:07<22:58, 28.72s/it]

...

[I 2026-04-05 21:43:43,888] Trial 41 finished with value: 3.093266410170489 and parameters: {'n_estimators': 311, 'max_depth': 30, 'max_features': None, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_samples': 0.6119665263472024}. Best is trial 47 with value: 3.0873333050729133.
🏃 View run serious-loon-306 at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5/runs/e04ebeb8a72c49bdaf311bbc75e944d2
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5


Best trial: 49. Best value: 3.08694: 100%|██████████| 50/50 [16:52<00:00, 20.26s/it]


[I 2026-04-05 21:43:52,249] Trial 49 finished with value: 3.0869398198716285 and parameters: {'n_estimators': 424, 'max_depth': 28, 'max_features': None, 'min_samples_split': 8, 'min_samples_leaf': 6, 'max_samples': 0.8542421813553751}. Best is trial 49 with value: 3.0869398198716285.


2026/04/05 21:45:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 21:45:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5/runs/b00cb95ef4224d738e69e52f46e37b51
🧪 View experiment at: https://dagshub.com/jay-kanakia/swiggy_delivery_time_prediction.mlflow/#/experiments/5


In [25]:
study.best_params

{'n_estimators': 424,
 'max_depth': 28,
 'max_features': None,
 'min_samples_split': 8,
 'min_samples_leaf': 6,
 'max_samples': 0.8542421813553751}

In [26]:
study.best_value

3.0869398198716285

In [27]:
optuna.visualization.plot_optimization_history(study)

In [29]:
optuna.visualization.plot_param_importances(study)

In [30]:
optuna.visualization.plot_slice(study)